# Journal Comparison and Migration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahulkaushal04/plotstyle/blob/main/examples/notebooks/03_journal_comparison_and_migration.ipynb)

When you resubmit a paper to a different journal, you need to update your figures to match the new journal's rules. PlotStyle makes this straightforward.

**What you will learn:**
- Compare two journals side by side with `plotstyle.diff()`
- See which journals are most similar to each other
- Automatically adapt a figure for a different journal with `plotstyle.migrate()`
- Browse any journal's full set of rules from the registry

In [ ]:
# Install plotstyle if needed (e.g., on Google Colab).
try:
    import plotstyle
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotstyle"])
    import plotstyle

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

import plotstyle

## 1. Compare Two Journals

`plotstyle.diff("nature", "science")` compares the rules for both journals and prints a table showing every setting that differs: figure dimensions, font sizes, export formats, and more.

In [ ]:
# Formatted table: Nature vs Science
result = plotstyle.diff("nature", "science")
print(result)

## 2. Access the Diff in Code

Besides printing the table, you can also work with the differences in code:

- `len(result)`: how many settings differ between the two journals
- `bool(result)`: `True` if there is at least one difference
- `result.differences`: the list of differences, each with a label and both values
- `result.to_dict()`: the full diff as a plain dictionary

In [ ]:
print(f"Number of differences: {len(result)}")
print(f"Specs differ: {bool(result)}\n")

# Iterate over each differing field
for d in result.differences:
    print(f"{d.label}:")
    print(f"  {result.journal_a}: {d.value_a}")
    print(f"  {result.journal_b}: {d.value_b}")

## 3. Compare Multiple Journals at Once

If you are deciding where to resubmit, it helps to know which journals require the fewest changes. Fewer differences means less work to adapt your figures.

In [ ]:
journals = ["nature", "science", "acs", "ieee", "plos"]

print(f"{'Pair':<22} {'Differences':>12}")
print("-" * 36)
for i, a in enumerate(journals):
    for b in journals[i + 1 :]:
        d = plotstyle.diff(a, b)
        print(f"{a} vs {b:<12} {len(d):>12}")

## 4. Create a Figure for Nature

Let's make a styled Nature figure. We will migrate it to Science in the next section and compare the sizes before and after.

In [ ]:
x = np.linspace(0, 10, 150)

with plotstyle.use("nature") as style:
    fig, ax = style.figure(columns=1)
    colors = style.palette(n=3)

    for i, c in enumerate(colors):
        ax.plot(x, np.sin(x + i), color=c, label=f"Dataset {i + 1}")

    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Intensity (a.u.)")
    ax.legend()
    ax.set_title("Before migration (Nature)")

w, h = fig.get_size_inches()
print(f"Nature figure size: {w:.2f} x {h:.2f} in")
plt.show()

## 5. Migrate to Science

`plotstyle.migrate()` updates the figure for the target journal in one step:
1. Resizes the figure to the new journal's column width
2. Scales all text proportionally so nothing is too small or too large
3. Applies the new journal's font and style settings

If anything significant changes (such as the font family or required resolution), a warning is printed.

> `migrate()` modifies the figure in-place. If you need to keep the original, save it first with `style.savefig()`.

In [ ]:
# Show all migration warnings in the output
warnings.filterwarnings("always")

plotstyle.migrate(fig, from_journal="nature", to_journal="science")

w, h = fig.get_size_inches()
print(f"Science figure size: {w:.2f} x {h:.2f} in")
ax.set_title("After migration (Science)")
plt.show()
plt.close(fig)

## 6. Inspect a Journal's Full Spec

You can look up all the rules for any journal directly from the registry. This is useful if you want to know the exact column widths, font sizes, DPI requirements, or export formats for a journal.

In [ ]:
spec = plotstyle.registry.get("nature")

print(f"Journal:          {spec.metadata.name}")
print(f"Publisher:        {spec.metadata.publisher}")
print(f"Single col:       {spec.dimensions.single_column_mm} mm")
print(f"Double col:       {spec.dimensions.double_column_mm} mm")
print(f"Font family:      {spec.typography.font_family}")
print(f"Font range:       {spec.typography.min_font_pt}-{spec.typography.max_font_pt} pt")
print(f"Panel labels:     {spec.typography.panel_label_pt} pt {spec.typography.panel_label_weight}")
print(f"Min DPI:          {spec.export.min_dpi}")
print(f"Preferred fmts:   {spec.export.preferred_formats}")
print(f"Colorblind req:   {spec.color.colorblind_required}")
print(f"Grayscale req:    {spec.color.grayscale_required}")

## 7. Compare Column Widths Across All Journals

This is a quick way to see how different journals size their figures. Single-column figures are typically used for smaller, simpler plots. Double-column figures span the full text width.

In [ ]:
print(f"{'Journal':<12} {'Single (mm)':<14} {'Double (mm)':<14}")
print("-" * 40)

for name in plotstyle.registry.list_available():
    s = plotstyle.registry.get(name)
    single = (
        f"{s.dimensions.single_column_mm}" if s.dimensions.single_column_mm is not None else "-"
    )
    double = (
        f"{s.dimensions.double_column_mm}" if s.dimensions.double_column_mm is not None else "-"
    )
    print(f"{name:<12} {single:<14} {double:<14}")